# NHL Draft Data — Exploratory Data Analysis

## Setup

In [37]:
import sqlite3
import pandas as pd

# connect to the database
conn = sqlite3.connect("../data/nhl_drafts_raw.db")

print("Connected!")

Connected!


In [38]:
# have pandas display all columns without truncating
pd.set_option("display.max_columns", None)

## Total Players Per Draft Year

In [39]:
# how many players per draft year?
pd.read_sql_query("""
    SELECT draft_year, COUNT(*) as total_picks
    FROM nhl_draft_picks
    GROUP BY draft_year
    ORDER BY draft_year
""", conn)

,draft_year,total_picks
0,2005,230
1,2006,213
2,2007,211
3,2008,211
4,2009,210
5,2010,210
6,2011,210
7,2012,211
8,2013,211
9,2014,210


All 13 draft years are present and accounted for. Pick totals are consistent 
in the 210-217 range for most years, with 2005 being a notable exception at 
230 due to additional compensatory picks awarded under the new post-lockout 
CBA. 2017 sits slightly higher at 217 reflecting the addition of the Vegas 
Golden Knights as an expansion team. No missing years, no unexpected outliers.

## Picks Per Round

In [40]:
# how many picks per round?
pd.read_sql_query("""
    SELECT round, COUNT(*) as total_picks
    FROM nhl_draft_picks
    GROUP BY round
    ORDER BY round
""", conn)

,round,total_picks
0,1,391
1,2,403
2,3,390
3,4,394
4,5,398
5,6,393
6,7,397


At first glance this looks slightly odd. With a set number of teams in the 
league, each assigned one pick per round per draft, we would expect the total 
number of picks in each round to be equal across all rounds.

The variation 
we see here warrants a closer look to confirm the data is correct and to 
understand what is driving the discrepancy.

### Deeper Investigation: Picks Per Round by Year


To understand where the discrepancies come from, we can break the pick counts 
down by draft year and round.

In [41]:
# number of picks per round in each draft
df = pd.read_sql_query("""
    SELECT draft_year, round, COUNT(*) as total_picks
    FROM nhl_draft_picks
    GROUP BY draft_year, round
    ORDER BY draft_year, round
""", conn)

pivot = df.pivot(index='draft_year', columns='round', values='total_picks')
pivot.columns = [f'Round {r}' for r in pivot.columns]

print(pivot.to_string())

            Round 1  Round 2  Round 3  Round 4  Round 5  Round 6  Round 7
draft_year                                                               
2005             30       31       30       34       37       32       36
2006             30       33       30       30       30       30       30
2007             30       31       30       30       30       30       30
2008             30       31       30       30       30       30       30
2009             30       31       30       29       30       30       30
2010             30       30       30       30       30       30       30
2011             30       31       29       30       30       30       30
2012             30       31       30       30       30       30       30
2013             30       31       30       30       30       30       30
2014             30       30       30       30       30       30       30
2015             30       31       30       30       30       30       30
2016             30       31       30 

This tells a much clearer story. Three distinct patterns emerge:

**2017 — Vegas Expansion (31 picks every round)**
The Vegas Golden Knights joined the NHL as an expansion team in 2017, 
bringing the league from 30 to 31 teams, explaining the uniform increase 
across all rounds.

**2005 — Post-Lockout CBA (230 total picks)**
The 2004-05 season was cancelled due to a lockout. The 2005 draft was the 
first under the new CBA and included additional compensatory picks, explaining 
the inflated totals particularly in rounds 4-7.

**2009 Round 4 & 2011 Round 3 — Forfeited Picks**
The Toronto Maple Leafs forfeited their 2009 4th round pick for an illegal 
bonus, and the New Jersey Devils forfeited their 2011 3rd round pick due to 
the Ilya Kovalchuk contract violation.

**Round 2 Compensatory Picks (most years)**
Almost every year shows 31 picks in Round 2. These are compensatory picks 
awarded to teams that failed to sign their former first round picks.

The table below details each compensatory pick awarded, and the unsigned 
first round pick that triggered it:

| Year | Team | Player Not Signed |
|------|------|-------------------|
| 2005 | LA Kings | David Steckel |
| 2006 | Boston | Lars Jonsson |
| 2006 | Edmonton | Jesse Niinimaki |
| 2006 | LA Kings | Jens Karlsson |
| 2007 | Anaheim | A.J. Thelen |
| 2008 | Phoenix | Blake Wheeler |
| 2009 | NY Rangers | Alexei Cherepanov |
| 2010 | No comp pick | — |
| 2011 | Montreal | David Fischer |
| 2012 | San Jose | Patrick White |
| 2013 | Winnipeg | Daultan Leveille |
| 2014 | No comp pick | — |
| 2015 | Chicago | Kevin Hayes |
| 2016 | Arizona | Conner Bleackley |
| 2017 | N/A — Vegas expansion bumped all rounds to 31 | — |

## Null Values Check

In [42]:
# nulls in key columns
pd.read_sql_query("""
    SELECT
        SUM(CASE WHEN player_name IS NULL THEN 1 ELSE 0 END) as missing_player_name,
        SUM(CASE WHEN position IS NULL THEN 1 ELSE 0 END) as missing_position,
        SUM(CASE WHEN drafted_by IS NULL THEN 1 ELSE 0 END) as missing_drafted_by,
        SUM(CASE WHEN games_played IS NULL THEN 1 ELSE 0 END) as missing_gp,
        SUM(CASE WHEN pick_number IS NULL THEN 1 ELSE 0 END) as missing_pick_number
    FROM nhl_draft_picks
""", conn)

,missing_player_name,missing_position,missing_drafted_by,missing_gp,missing_pick_number
0,0,0,0,1364,0


No missing values in the key identifying columns. The 1,364 missing values 
in games_played are expected and not a data quality issue — they simply 
represent players who were drafted but never played a regular season NHL game. 
This will be an important consideration during analysis.

## Duplicates Check

In [43]:
# duplicates check
pd.read_sql_query("""
    SELECT player_name, draft_year, COUNT(*) as count
    FROM nhl_draft_picks
    GROUP BY player_name, draft_year
    HAVING count > 1
""", conn)

,player_name,draft_year,count


No duplicate entries found. Every player appears exactly once per draft year.

## Position Distribution

In [44]:
# position distriubtion
pd.read_sql_query("""
    SELECT position, COUNT(*) as count
    FROM nhl_draft_picks
    GROUP BY position
    ORDER BY count DESC
""", conn)

,position,count
0,D,935
1,C,617
2,L,480
3,R,385
4,G,282
5,F,55
6,W,11
7,,1


The majority of positions are labeled consistently using standard NHL 
designations: D (Defenseman), C (Center), L (Left Wing), R (Right Wing), 
and G (Goalie).

However 55 players are listed as F (Forward) and 11 as W 
(Wing), which are less specific but not incorrect — they simply reflect 
inconsistent labeling in the source data. Since positional breakdown is not central to this analysis, 
we will not be investigating draft success by specific position.

One player has no position listed at all, which is a true missing value that will be addressed in the cleaning 
step below.

### Fix: Kyell Henegan — Missing Position

In [45]:
# find positionless player
pd.read_sql_query("""
    SELECT player_name, draft_year, pick_number, drafted_by, position
    FROM nhl_draft_picks
    WHERE position = '' OR position IS NULL
""", conn)

,player_name,draft_year,pick_number,drafted_by,position
0,Kyell Henegan,2006,208,New Jersey,


A search shows us that the player without a position is Kyell Henegan. Searching him online reveals that he is a Defenseman.

We will assign him the position 'D' and create a new cleaned version of the dataset with this update.

In [46]:
import shutil
import os

# remove existing clean db if it exists so we always start fresh
if os.path.exists("../data/nhl_drafts_clean.db"):
    os.remove("../data/nhl_drafts_clean.db")

# copy raw db to clean db
shutil.copy("../data/nhl_drafts_raw.db", "../data/nhl_drafts_clean.db")

# connect to the clean db
conn_clean = sqlite3.connect("../data/nhl_drafts_clean.db")
print("Connected!")

Connected!


In [47]:
# fix Kyell Henegan's missing position
conn_clean.execute("""
    UPDATE nhl_draft_picks
    SET position = 'D'
    WHERE player_name = 'Kyell Henegan'
    AND draft_year = 2006
""")

conn_clean.commit()

In [48]:
# verify the fix
pd.read_sql_query("""
    SELECT player_name, draft_year, pick_number, drafted_by, position
    FROM nhl_draft_picks
    WHERE player_name = 'Kyell Henegan'
""", conn_clean)

,player_name,draft_year,pick_number,drafted_by,position
0,Kyell Henegan,2006,208,New Jersey,D


Position confirmed as 'D'. The only true missing value has been resolved.

### Fix: Comlete Team Names

The original dataset shortens team names to the city, so we will replace these with the official full team names.

In [49]:
team_name_map = {
    'Anaheim': 'Anaheim Ducks',
    'Arizona': 'Arizona Coyotes',
    'Atlanta': 'Atlanta Thrashers',
    'Boston': 'Boston Bruins',
    'Buffalo': 'Buffalo Sabres',
    'Calgary': 'Calgary Flames',
    'Carolina': 'Carolina Hurricanes',
    'Chicago': 'Chicago Blackhawks',
    'Colorado': 'Colorado Avalanche',
    'Columbus': 'Columbus Blue Jackets',
    'Dallas': 'Dallas Stars',
    'Detroit': 'Detroit Red Wings',
    'Edmonton': 'Edmonton Oilers',
    'Florida': 'Florida Panthers',
    'Los Angeles': 'Los Angeles Kings',
    'Minnesota': 'Minnesota Wild',
    'Montreal': 'Montreal Canadiens',
    'NY Islanders': 'New York Islanders',
    'NY Rangers': 'New York Rangers',
    'Nashville': 'Nashville Predators',
    'New Jersey': 'New Jersey Devils',
    'Ottawa': 'Ottawa Senators',
    'Philadelphia': 'Philadelphia Flyers',
    'Phoenix': 'Phoenix Coyotes',
    'Pittsburgh': 'Pittsburgh Penguins',
    'San Jose': 'San Jose Sharks',
    'St. Louis': 'St. Louis Blues',
    'Tampa Bay': 'Tampa Bay Lightning',
    'Toronto': 'Toronto Maple Leafs',
    'Vancouver': 'Vancouver Canucks',
    'Vegas': 'Vegas Golden Knights',
    'Washington': 'Washington Capitals',
    'Winnipeg': 'Winnipeg Jets'
}

# standardize team names to full franchise names
conn_clean.execute("""
    UPDATE nhl_draft_picks
    SET drafted_by = CASE drafted_by
        WHEN 'Anaheim' THEN 'Anaheim Ducks'
        WHEN 'Arizona' THEN 'Arizona Coyotes'
        WHEN 'Atlanta' THEN 'Atlanta Thrashers'
        WHEN 'Boston' THEN 'Boston Bruins'
        WHEN 'Buffalo' THEN 'Buffalo Sabres'
        WHEN 'Calgary' THEN 'Calgary Flames'
        WHEN 'Carolina' THEN 'Carolina Hurricanes'
        WHEN 'Chicago' THEN 'Chicago Blackhawks'
        WHEN 'Colorado' THEN 'Colorado Avalanche'
        WHEN 'Columbus' THEN 'Columbus Blue Jackets'
        WHEN 'Dallas' THEN 'Dallas Stars'
        WHEN 'Detroit' THEN 'Detroit Red Wings'
        WHEN 'Edmonton' THEN 'Edmonton Oilers'
        WHEN 'Florida' THEN 'Florida Panthers'
        WHEN 'Los Angeles' THEN 'Los Angeles Kings'
        WHEN 'Minnesota' THEN 'Minnesota Wild'
        WHEN 'Montreal' THEN 'Montreal Canadiens'
        WHEN 'NY Islanders' THEN 'New York Islanders'
        WHEN 'NY Rangers' THEN 'New York Rangers'
        WHEN 'Nashville' THEN 'Nashville Predators'
        WHEN 'New Jersey' THEN 'New Jersey Devils'
        WHEN 'Ottawa' THEN 'Ottawa Senators'
        WHEN 'Philadelphia' THEN 'Philadelphia Flyers'
        WHEN 'Phoenix' THEN 'Phoenix Coyotes'
        WHEN 'Pittsburgh' THEN 'Pittsburgh Penguins'
        WHEN 'San Jose' THEN 'San Jose Sharks'
        WHEN 'St. Louis' THEN 'St. Louis Blues'
        WHEN 'Tampa Bay' THEN 'Tampa Bay Lightning'
        WHEN 'Toronto' THEN 'Toronto Maple Leafs'
        WHEN 'Vancouver' THEN 'Vancouver Canucks'
        WHEN 'Vegas' THEN 'Vegas Golden Knights'
        WHEN 'Washington' THEN 'Washington Capitals'
        WHEN 'Winnipeg' THEN 'Winnipeg Jets'
    END
""")

conn_clean.commit()

# verify
pd.read_sql_query("""
    SELECT DISTINCT drafted_by FROM nhl_draft_picks ORDER BY drafted_by
""", conn_clean)

,drafted_by
0,Anaheim Ducks
1,Arizona Coyotes
2,Atlanta Thrashers
3,Boston Bruins
4,Buffalo Sabres
5,Calgary Flames
6,Carolina Hurricanes
7,Chicago Blackhawks
8,Colorado Avalanche
9,Columbus Blue Jackets


## NHL Reach Rate

In [50]:
# what percentage of drafted players actually made the NHL?
pd.read_sql_query("""
    SELECT
        COUNT(*) as total_drafted,
        SUM(CASE WHEN games_played > 0 THEN 1 ELSE 0 END) as reached_nhl,
        ROUND(100.0 * SUM(CASE WHEN games_played > 0 THEN 1 ELSE 0 END) / COUNT(*), 1) as pct_reached_nhl
    FROM nhl_draft_picks
""", conn_clean)

,total_drafted,reached_nhl,pct_reached_nhl
0,2766,1402,50.7


Just over half of all drafted players in this dataset went on to play at 
least one NHL regular season game. This 50.7% figure sets an important 
baseline for the analysis — being drafted is far from a guarantee of reaching 
the NHL, and the further down the draft order a player is selected, the less 
likely they are to make it. This will be a key theme in the analysis.

## Top 10 Players by Points — Sanity Check

In [51]:
# top 10 players by points
pd.read_sql_query("""
    SELECT player_name, draft_year, pick_number, drafted_by, points
    FROM nhl_draft_picks
    ORDER BY points DESC
    LIMIT 10
""", conn_clean)

,player_name,draft_year,pick_number,drafted_by,points
0,Sidney Crosby,2005,1,Pittsburgh Penguins,1756
1,Patrick Kane,2007,1,Chicago Blackhawks,1393
2,Anze Kopitar,2005,11,Los Angeles Kings,1314
3,Steven Stamkos,2008,1,Tampa Bay Lightning,1250
4,Connor McDavid,2015,1,Edmonton Oilers,1208
5,John Tavares,2009,1,New York Islanders,1182
6,Claude Giroux,2006,22,Philadelphia Flyers,1160
7,Nathan MacKinnon,2013,1,Colorado Avalanche,1137
8,Nikita Kucherov,2011,58,Tampa Bay Lightning,1119
9,Leon Draisaitl,2014,3,Edmonton Oilers,1053


The top 10 players by career points are exactly who you would expect to see, 
providing a strong gut check that the data is accurate. Sidney Crosby leads 
with 1,746 points, followed by Patrick Kane and Anze Kopitar. Notably Nikita 
Kucherov appears at pick 58 in round 2 of the 2011 draft — a reminder that 
elite talent is not always found at the top of the draft, which is central 
to the question this project aims to investigate.

## End of EDA

All discrepancies in the data are accounted for by real world NHL events. The raw dataset has been validated and a cleaned version saved to data/nhl_drafts_clean.db. This file will be used for all subsequent analysis.

In [52]:
conn.close()
conn_clean.close()
print("All database connections closed.")

All database connections closed.
